In [9]:
# ── Imports ───────────────────────────────────────────────────────────────
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import datetime
import warnings

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    warnings.warn('wandb not installed — logging disabled.')

from envs.env import FoodEnv
from agents.ppo import PPOAgent
from models.model import *
from envs.env_config import NUTRIENT_CONFIG, DISCRETE_EAT_AMOUNT

now           = datetime.datetime.now()
curr_date_time = 'd_' + now.strftime('%d_%m_%Y') + '_t_' + now.strftime('%H_%M_%S')
print('Run timestamp:', curr_date_time)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED  =  2026  #100 # 42, 0, 27, 64, 107, 7 

# seed 0 fails from 1600 ep
# seed 42 fails from 800 ep

# With lr scheduler
# seed 0 fails around 1k ep

Run timestamp: d_26_07_2026_t_11_18_24


In [10]:
# ── WandB initialisation ──────────────────────────────────────────────────
PROJECT_NAME = 'Food RL'
RUN_NAME     = 'Run_Seed'+ str(SEED)
log_wandb    = True   # set True to enable wandb logging

if log_wandb and WANDB_AVAILABLE:
    print(f'Logging to wandb: project={PROJECT_NAME}  run={RUN_NAME}  time={curr_date_time}')
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)
else:
    print('WandB logging disabled.')


Logging to wandb: project=Food RL  run=Run_Seed2026  time=d_26_07_2026_t_11_18_24


In [11]:
# ── Config ────────────────────────────────────────────────────────────────
FOOD_FOLDER = 'food_dataset'
PLOT_DIR = 'results/plots'
MENU_SIZE   = 1


# ── Action space mode ─────────────────────────────────────────────────────
# True  -> Box(num_foods,) continuous: fractional consumption per slot.
# False -> Discrete(num_foods+1): at most ONE slot eaten per step at the fixed
#          amount env_config.DISCRETE_EAT_AMOUNT.
IS_CONTINUOUS = False

# ── Network architecture ──────────────────────────────────────────────────
HIDDEN = 256
SHARED = True     # True  -> one SharedActorCritic trunk feeds both heads
                  # False -> independent Actor + Critic (each its own trunk)

# ── Bottleneck configuration (ONE nested dict) ────────────────────────────
# Replaces the old scattered BOTTLENECK_DIM / SEPARATE_BRANCHES / *_HEAD_*
# flags. Spec language:
#     trunk spec : None | int | {"phy": int|None, "food": int|None}
#     head spec  : None | int          (None / omitted = no bottleneck there)
#
# SHARED=True  -> {"trunk": <trunk>, "actor_head": <head>, "critic_head": <head>}
# SHARED=False -> {"actor":  {"trunk": <trunk>, "head": <head>},
#                  "critic": {"trunk": <trunk>, "head": <head>}}
#
# In non-shared mode the actor and critic trunks are configured INDEPENDENTLY
# (they can even have different dims). The names you can probe later with
# agent.set_amplitude(name, gain) — call agent.list_bottlenecks() for the
# live set:
#   shared     : trunk / trunk_phy / trunk_food / actor_head / critic_head
#   non-shared : actor_trunk[_phy/_food] / actor_head /
#                critic_trunk[_phy/_food] / critic_head
if SHARED:
    BOTTLENECKS = {
        "trunk":       {"phy": 16, "food": None},   # separate phy/food branches
        "actor_head":  None,
        "critic_head": 16,
    }
else:
    BOTTLENECKS = {
        "actor":  {"trunk": {"phy": 16, "food": None}, "head": None},
        "critic": {"trunk": {"phy": 16, "food": None}, "head": 16},
    }

# Amplitude note: training always starts neutral (gain 1.0 everywhere).
# Per-neuron amplitude probing is a POST-hoc tool applied at inference time
# via agent.set_amplitude(name, vector) — see the Inference amplitude cell.

# ── Sleep / wake cycle ────────────────────────────────────────────────────
AWAKE_STEPS = 96 #192 #480
SLEEP_STEPS = 48 #96 #240
NUM_CYCLES  = 3
MAX_STEPS   = NUM_CYCLES * (AWAKE_STEPS + SLEEP_STEPS)

os.makedirs(PLOT_DIR, exist_ok=True)
print(f'Episode length: {MAX_STEPS} steps  ({NUM_CYCLES} cycles)')
print(f'Action mode: {"continuous" if IS_CONTINUOUS else "discrete"}')
print(f'Architecture: shared={SHARED}  hidden={HIDDEN}')
print(f'Bottlenecks config: {BOTTLENECKS}')


Episode length: 432 steps  (3 cycles)
Action mode: discrete
Architecture: shared=True  hidden=256
Bottlenecks config: {'trunk': {'phy': 16, 'food': None}, 'actor_head': None, 'critic_head': 16}


In [12]:
# ── Environment ───────────────────────────────────────────────────────────
# NUTRIENT_CONFIG and DISCRETE_EAT_AMOUNT now live in env_config.py, not
# here — edit that file directly to add/change nutrients or the fixed
# discrete eating amount. Nothing in this cell needs to change for that.
env = FoodEnv(
    food_folder=FOOD_FOLDER,
    num_foods=MENU_SIZE,
    max_steps=MAX_STEPS,
    one_hot_embedding=True,
    seed=SEED,
    consumption_threshold=0.0,#0.1,    # amounts below this → zero (no absorption)
    awake_steps_per_cycle=AWAKE_STEPS,
    sleep_steps_per_cycle=SLEEP_STEPS,
    is_continuous=IS_CONTINUOUS,   # <- CHANGE Cell 2's IS_CONTINUOUS, not here
)

print('\n── Environment summary ──────────────────────────────────────')
print(f'  Food items             : {env.num_items}')
print(f'  Nutrients              : {env.num_nutrients}  {env.nutrient_names}')
print(f'  Observation state dim  : {env.state_dim}  (nutrients + is_awake + time_in_cycle)')
print(f'  Menu size              : {env.num_foods}')
print(f'  Action space           : {env.action_space}')
print(f'  Max steps               : {env.max_steps}')
print(f'  Awake steps / cycle    : {env.awake_steps}')
print(f'  Sleep steps / cycle    : {env.sleep_steps}')
print(f'  Cycle length           : {env.cycle_length}')
print(f'  Consumption threshold  : {env.consumption_threshold}')
print(f'  Target (normed)        : {env._norm_targets}')
print(f'  Target low             : {env._norm_target_low}')
print(f'  Target high            : {env._norm_target_high}')
print()
print(env.nutrient_norm_summary().to_string(index=False))
print('─────────────────────────────────────────────────────────────')


[FoodEnv] Loading  'glucose'  from  'food_dataset/serum_glucose.csv'
           min=70.0758  max=142.5123   foods=19   time_points=500
[FoodEnv] Loading  'peptides'  from  'food_dataset/small_peptides_absorbed.csv'
           min=0.0000  max=0.0018   foods=19   time_points=500
[FoodEnv] Loading  'fatty_acids'  from  'food_dataset/fatty_acids_absorbed.csv'
           min=0.0000  max=0.0007   foods=19   time_points=500
[FoodEnv] Ready — 19 foods | 3 time-series nutrients | 0 cumulative nutrients
[FoodEnv] Loading shadow nutrient 'fullness'  from  'food_dataset/fullness.csv'
[FoodEnv] Loading shadow nutrient 'hunger'  from  'food_dataset/hunger.csv'
[FoodEnv] Loading shadow nutrient 'cck'  from  'food_dataset/cck.csv'
[FoodEnv] Loading shadow nutrient 'ghrelin'  from  'food_dataset/ghrelin.csv'
[FoodEnv] Loading shadow nutrient 'glp_1'  from  'food_dataset/glp_1.csv'
[FoodEnv] Loading shadow nutrient 'pyy'  from  'food_dataset/pyy.csv'
[FoodEnv] Shadow nutrients ready — 6 signal(s): ['ful

In [13]:
# ── Agent ─────────────────────────────────────────────────────────────────
# Network construction happens INSIDE PPOAgent — it reads env.is_continuous
# and the unified `bottlenecks` config to build the right classes automatically.
agent = PPOAgent(
    env, device='cpu', seed=SEED, limit_delta=1.0,
    shared=SHARED,
    hidden=HIDDEN,
    bottlenecks=BOTTLENECKS,
)

# ── To inject a custom network instead (advanced) ────────────────────────
# num_foods      = env.num_foods
# state_shape    = env.observation_space["physiological_state"].shape[0]
# food_emb_shape = env.observation_space["food_embeddings"].shape
# food_flat_size = food_emb_shape[0] * food_emb_shape[1]
# actor_cls = Actor if env.is_continuous else DiscreteActor
# actor  = actor_cls(state_size=state_shape, food_flat_size=food_flat_size,
#                    num_foods=num_foods, seed=SEED, hidden=HIDDEN,
#                    trunk={"phy": 16, "food": None}, head=None)
# critic = Critic(state_size=state_shape, food_flat_size=food_flat_size,
#                 seed=SEED, hidden=HIDDEN, trunk=6, head=16)
# agent = PPOAgent(env, device='cpu', shared=False, seed=SEED, limit_delta=1.0,
#                  actor_network=actor, critic_network=critic)

print(f'Agent built. shared={SHARED}  is_continuous={agent.is_continuous}')
if SHARED:
    print(f'  policy: {type(agent.policy).__name__}')
else:
    print(f'  actor:  {type(agent.actor).__name__}')
    print(f'  critic: {type(agent.critic).__name__}')
print('Bottlenecks:', agent.list_bottlenecks())


Agent built. shared=True  is_continuous=False
  policy: SharedDiscreteActorCritic
Bottlenecks: {'trunk_phy': {'dim': 16, 'amplitude': 1.0, 'bias': 0.0}, 'critic_head': {'dim': 16, 'amplitude': 1.0, 'bias': 0.0}}


In [14]:
# ── Train ─────────────────────────────────────────────────────────────────
NUM_EPISODES = 2000 #10000 #150
print(f'Starting training: {NUM_EPISODES} episodes …')


training_log = agent.train(
    num_episodes=NUM_EPISODES,
    rollout_steps=256, #512,
    ppo_epochs=4,
    minibatch_size=64,
    log_every_episodes=2,
    printing=True,
    # actor_lr=5e-6, # 1e-5
    # critic_lr=1e-2,
    shared_ac_lr = 1e-4,
    log_wandb=log_wandb,
    checkpoint_every=250,        # save a bundle every N episodes
        # -> per-seed folder 'checkpoints_<SEED>/' (e.g. checkpoints_42/);
        #    the best single-episode bundle is saved there automatically too
    lr_schedule={"type": "exponential", "final_frac": 0.1},
    checkpoint_dir='results/checkpoints',
)



# train() returns the per-episode training log (dict of equal-length arrays):
#   reward, distance, consumption (food-item COUNT), actor_loss, critic_loss.
returns     = training_log['reward']
consumption = training_log['consumption']      # food items eaten per episode
distance    = training_log['distance']

print(f'\nDone.')
print(f'  Final rolling avg return     (last 50): {np.mean(returns[-50:]):.3f}')
print(f'  Final rolling avg food items (last 50): {np.mean(consumption[-50:]):.3f}')
print(f'  Final rolling avg distance   (last 50): {np.mean(distance[-50:]):.3f}')


Starting training: 2000 episodes …
  [best] new best return -1711.62 @ ep1 → results/checkpoints_2026/ppo_agent_best_*
  [best] new best return -844.39 @ ep2 → results/checkpoints_2026/ppo_agent_best_*
Episode     2 | Last Return:  -844.39 | Rolling Avg(50): -1278.00 | Food items:  105 | Total Distance:  1373.49 | Actor L:  -0.013 | Critic L: 1667.009
  [best] new best return -521.20 @ ep3 → results/checkpoints_2026/ppo_agent_best_*
  [best] new best return -507.94 @ ep4 → results/checkpoints_2026/ppo_agent_best_*
Episode     4 | Last Return:  -507.94 | Rolling Avg(50):  -896.29 | Food items:   73 | Total Distance:  1073.41 | Actor L:  -0.013 | Critic L: 413.858
  [best] new best return -68.05 @ ep5 → results/checkpoints_2026/ppo_agent_best_*
  [best] new best return -15.25 @ ep6 → results/checkpoints_2026/ppo_agent_best_*
Episode     6 | Last Return:   -15.25 | Rolling Avg(50):  -611.41 | Food items:   37 | Total Distance:   406.39 | Actor L:  -0.008 | Critic L:   8.784
  [best] new b

**Note on `actor_lr=1e-5` vs `critic_lr=1e-2`:** this is a 1000x gap, larger
than typical PPO setups (usually both within the same order of magnitude,
e.g. both ~3e-4). At `1e-5` the actor may move very slowly over even 10,000
episodes — if the return curve below looks flat or the agent barely eats,
try narrowing this gap (e.g. `actor_lr=3e-4, critic_lr=1e-3`) before
concluding anything from training or from the latent analysis later in this
notebook. A policy that hasn't learned much will not show a meaningful
latent signature no matter how the bottleneck or PCA is configured.


In [15]:
# ── Save model + training curves (training -> inference handoff) ──────────
# The ONLY handoff point between Training and Inference. save_final() writes a
# self-contained bundle under results/checkpoints_{SEED}/:
#     ppo_agent_last_shared.pt      (or _actor.pt / _critic.pt)
#     ppo_agent_last_meta.json      (full env + agent config — enough to
#                                    rebuild the env/agent and run inference)
#     ppo_agent_last_log.npz        (per-episode reward/distance/consumption/
#                                    actor_loss/critic_loss)
# The BEST single-xepisode checkpoint (ppo_agent_best_*) is saved automatically
# DURING training whenever checkpoint_every is set (see the Train cell), in the
# same seed folder.
# save_final() creates the per-seed folder (results/checkpoints_{SEED}/) itself.
last_path = agent.save_final('results/checkpoints', log_wandb=log_wandb)

# Standalone per-episode training log for later paper plots. Keys:
#   reward, distance, consumption (food-item count), actor_loss, critic_loss.
agent.save_training_log(f'results/checkpoints_{SEED}/training_log.npz')

print('Saved bundle to', last_path + '_*')
print('meta.json path:', last_path + '_meta.json')
print('Reload in Inference with PPOAgent.from_checkpoint(<that meta.json>).')


Saved bundle to results/checkpoints_2026/ppo_agent_last_*
meta.json path: results/checkpoints_2026/ppo_agent_last_meta.json
Reload in Inference with PPOAgent.from_checkpoint(<that meta.json>).


In [16]:
# ── Finish wandb run (Training cleanup) ───────────────────────────────────
# Closes out the SAME wandb run opened in the 'WandB initialisation' cell
# above (log_wandb / wandb.init are training-side concerns) — this stays in
# Training, not Inference, since Inference never opens a wandb run of its
# own and has no `log_wandb` flag to check.
if log_wandb and WANDB_AVAILABLE:
    wandb.finish()
    print('WandB run finished.')


train/actor_loss,▇▆▁▆▆▆▅▆▆▇▆▅▆▇█▇█▅▆▅▆▇▇█▇█▇▇▇▇█▇█▇▇█▇▇▇▇
train/consumption,█▁▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
train/critic_loss,▃▂▆█▁▂▂▂▃▂▄▂▁▁▂▃▄▁▂▂▂▄▂▁▂▂▂▂▂▂▁▂▂▁▂▁▂▁▂▁
train/distance,█▂█▃▂▄▂▂▃▃▁▂▂▂▁▂▁▂▂▂▁▂▂▁▁▂▁▂▂▁▂▁▁▂▁▁▁▂▂▁
train/entropy,█▆▇▃▅▆█▅▅▇▄▃▃▂▂▄▂▄▄▂▄▄▂▂▂▂▂▂▃▁▂▁▃▂▂▂▂▂▂▂
train/reward,▁▂▃▄▅▅▃▆▄▇▇▆▅▅▅▆██▇▆▇▇▇▆▇█▆▅▆▇▆▆▆▆█▇▆▆▇▇
train/reward_rolling_avg,▁▂▇▇████████████████████████████████████
train/shared_ac_lr,██▇▇▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
train/actor_loss,-0.00132
train/consumption,12
train/critic_loss,2.36552


WandB run finished.
